# Ausgewogene BSTs

## Agenda

1. Motivation  
2. Definition von „Balance“  
3. AVL-Bäume  
5. Andere ausgewogene Bäume

## 1. Motivation

In [ ]:
class BSTree:
    class Node:
        def __init__(self, val, left=None, right=None):
            self.val = val
            self.left = left
            self.right = right
            
    def __init__(self):
        self.size = 0
        self.root = None
    
    def insert(self, val):
        assert val not in self
        def insert_rec(node):
            if not node:
                return BSTree.Node(val)
            elif val < node.val:
                node.left = insert_rec(node.left)
            else:
                node.right = insert_rec(node.right)
            return node
        self.root = insert_rec(self.root)
        self.size += 1

    def __contains__(self, val):
        def contains_rec(node):
            if not node:
                return False
            elif val == node.val:
                return True
            elif val < node.val:
                return contains_rec(node.left)
            else:
                return contains_rec(node.right)
        return contains_rec(self.root)        
                        
    def pprint(self, width=64):
        height = self.height()
        nodes  = [(self.root, 0)]
        prev_level = 0
        repr_str = ''
        while nodes:
            n,level = nodes.pop(0)
            if prev_level != level:
                prev_level = level
                repr_str += '\n'
            if not n:
                if level < height-1:
                    nodes.extend([(None, level+1), (None, level+1)])
                repr_str += '{val:^{width}}'.format(val='-', width=width//2**level)
            elif n:
                if n.left or level < height-1:
                    nodes.append((n.left, level+1))
                if n.right or level < height-1:
                    nodes.append((n.right, level+1))
                repr_str += '{val:^{width}}'.format(val=n.val, width=width//2**level)
        print(repr_str)
    
    def height(self):
        def height_rec(t):
            if not t:
                return 0
            else:
                return max(1+height_rec(t.left), 1+height_rec(t.right))
        return height_rec(self.root)

In [ ]:

t = BSTree()
for x in range(6):
    t.insert(x)
t.pprint()

Wenn ein binärer Suchbaum nicht balanciert ist, kann er sich zu einer verketteten Liste verschlechtern! Dies führt zu einer Zeitkomplexität von $O(n)$ für alle Operationen.

## 2. Definition von „Balance“

Es gibt verschiedene Kriterien, die wir verwenden können, um zu entscheiden, ob ein binärer Baum „balanciert“ ist, z.B.:

- die *Anzahl* der Knoten auf jeder Seite eines gegebenen Knotens
- die *Höhe* der Teilbäume auf jeder Seite eines gegebenen Knotens
- die *Dichte* der Knoten auf jeder Seite eines gegebenen Knotens
- die *Form* der Teilbäume auf jeder Seite eines gegebenen Knotens (vielleicht wollen wir nur vollständige oder komplette Bäume zulassen).

Was sind die Vor- und Nachteile einiger dieser Kriterien?

### 2.1. Balancierfaktor

*def.* In einem höhenbalancierten Baum ist der *Balancierfaktor* eines Knotens die Höhe seines rechten Teilbaums minus der Höhe seines linken Teilbaums.

![](../images/16-bf-quiz.png)

Um die Balancefaktoren effizient zu berechnen, müssen wir schnell auf die Höhe jedes Knotens im Baum zugreifen können. Wir können es uns nicht leisten, die Höhe eines Teilbaums jedes Mal neu zu berechnen, wenn wir sie benötigen!

In [ ]:
# Update the BSTree class so that:
# - the node class has a height attribute
# - the height of a node is updated during the `insert` operation
#   (do this without recalculating the height of sub-trees)

class BSTree(BSTree):
    class Node:
        def __init__(self, val, left=None, right=None):
            self.val = val
            self.left = left
            self.right = right
            self.height = 1

    @staticmethod
    def update_height(n: Node):
        pass
            
    def insert(self, val):
        assert val not in self
        def insert_rec(node):
            if not node:
                return BSTree.Node(val)
            elif val < node.val:
                node.left = insert_rec(node.left)
                self.update_height(node)
            else:
                node.right = insert_rec(node.right)
                self.update_height(node)
            return node
        self.root = insert_rec(self.root)
        self.size += 1

    # updated pprint shows balance factors calculated from node heights
    def pprint(self, width=64):
        height = self.root.height if self.root else 0
        nodes  = [(self.root, 0)]
        prev_level = 0
        repr_str = ''
        while nodes:
            n,level = nodes.pop(0)
            if prev_level != level:
                prev_level = level
                repr_str += '\n'
            if not n:
                if level < height-1:
                    nodes.extend([(None, level+1), (None, level+1)])
                repr_str += f'{"-":^{width//2**level}}'
            elif n:
                if n.left or level < height-1:
                    nodes.append((n.left, level+1))
                if n.right or level < height-1:
                    nodes.append((n.right, level+1))
                bf = ((n.right.height if n.right else 0) 
                      - (n.left.height if n.left else 0))
                repr_str += f'{f"{n.val}[{bf}]":^{width//2**level}}'
        print(repr_str)

In [ ]:
import random
vals = list(range(10))
random.shuffle(vals)

t = BSTree()
for x in vals:
    t.insert(x)
t.pprint()

## 3. AVL (Adelson-Velsky und Landis) Bäume

*def.* Ein AVL-Baum ist ein höhenbalancierter binärer Suchbaum, bei dem der Balancefaktor für jeden Knoten entweder -1, 0 oder 1 ist. Wir nennen dies die *AVL-Eigenschaft*.

Ein AVL-Baum ist *selbstbalancierend*. Das heißt, immer wenn die AVL-Eigenschaft durch eine Einfügung oder Löschung verletzt werden könnte, müssen wir den Baum vor dem Beenden der Operation wieder in Ordnung bringen.

Wann kann die AVL-Eigenschaft durch Einfügungen/Löschungen verletzt werden?

![](../images/16-avl-violations.png)

Wie können wir diese Verletzungen beheben, wenn sie auftreten?

### 3.1. Wesentliche Operation: „Rotation“

Wenn der Balancefaktor eines Knotens in einem binären Suchbaum $< -1$ (linksüberlastet) oder $> 1$ (rechtsüberlastet) ist, benötigen wir eine Operation, um die Knoten im Baum neu zu verteilen und das Gleichgewicht wiederherzustellen.

Die Operation muss:
1. die Eigenschaft des binären Suchbaums erhalten
2. die Höhen und Balancefaktoren der beteiligten Knoten vorhersehbar verändern

Die Lösung ist ein symmetrisches Paar von Operationen, die „Rotationen“ genannt werden.

![](../images/16-rotations.png)

In [ ]:
# Implement in-place right rotation for the AVL tree node
# - be sure to update the heights of the nodes after rotation!

class AVLTree(BSTree):
    class Node:
        def __init__(self, val, left=None, right=None):
            self.val = val
            self.left = left
            self.right = right
            self.height = 1

        def rotate_right(self):
            pass
    
    def insert(self, val):
        assert val not in self
        def insert_rec(node):
            if not node:
                return AVLTree.Node(val)
            elif val < node.val:
                node.left = insert_rec(node.left)
                self.update_height(node)
            else:
                node.right = insert_rec(node.right)
                self.update_height(node)
            return node
        self.root = insert_rec(self.root)
        self.size += 1

In [ ]:
t = AVLTree()
for x in range(6, 0, -1):
    t.insert(x)
t.pprint()

In [ ]:
t.root.rotate_right()
t.pprint()

In [ ]:
t.root.rotate_right()
t.pprint()

In [ ]:
t.root.left.rotate_right()
t.pprint()

Beachte, dass Rotationen *nicht immer* die Höhe oder den Balancefaktor von Knoten auf nützliche Weise verändern! 

![](../images/16-right-rotations.png)

### 3.2. „Aus dem Gleichgewicht“ Szenarien & Rotationsrezepte

Es gibt eine begrenzte Anzahl von Möglichkeiten, wie ein Knoten aus dem Gleichgewicht geraten kann. Für jedes dieser Szenarien gibt es ein entsprechendes „Rezept“, das das Gleichgewicht wiederherstellt. Überzeuge dich selbst davon, dass diese Rezepte funktionieren!

![](../images/16-rotation-recipes.png)

### 3.3. Einfügen & Ausbalancieren

Das Einfügen in einen AVL-Baum kann eine Unausgewogenheit verursachen, die genau **eine Anwendung eines Rotationsrezepts** (d.h. 1-2 Rotationen) zur Behebung erfordert.

In [ ]:
# Implement `rebalance` to deal with the "LL" case

class AVLTree(AVLTree):
    @staticmethod
    def rebalance(node):
        pass
            
    def insert(self, val):
        assert val not in self
        def insert_rec(node):
            if not node:
                return AVLTree.Node(val)
            elif val < node.val:
                node.left = insert_rec(node.left)
                self.update_height(node)
            else:
                node.right = insert_rec(node.right)
                self.update_height(node)

            # potentially rebalance
            self.rebalance(node)
            return node
        self.root = insert_rec(self.root)
        self.size += 1

In [ ]:
val = 50
t = AVLTree()

In [ ]:
# (evaluate multiple times with ctrl-enter)
t.insert(val)
val -= 1
t.pprint()

### 3.4. Löschen und Ausbalancieren

Löschungen können im Gegensatz zu Einfügungen ein Ungleichgewicht verursachen, das **mehrfache Anwendungen eines Rotationsrezepts** zur Behebung erfordert! Diese Ungleichgewichte, falls vorhanden, werden bei den Vorfahren des gelöschten Knotens gefunden.

![](../images/16-deletions.png)

### 3.5. Analyse des AVL-Baums

Zentral für die Laufzeitkomplexität der Operationen in einem AVL-Baum ist die Höhe des Baums $h$ in Bezug auf die Anzahl der Knoten $n$ im Baum.

Wir definieren $M(h)$ als die *minimale Anzahl von Knoten* in einem AVL-Baum der Höhe $h$. Man kann sich dies als den am *dünnsten besiedelten* oder *Worst-Case*-AVL-Baum der Höhe $h$ vorstellen.

Es sollte klar sein, dass $M(1) = 1$ und $M(2) = 2$ gilt. (Zeichnen Sie diese Bäume!)

Ein AVL-Baum der Höhe $=3$ mit minimaler Knotenzahl hat einen Wurzelknoten und Teilbäume der Höhe $M(1)$ und $M(2)$, also gilt $M(3) = 1 + M(1) + M(2)$.

Allgemein können wir sagen, dass $M(h) = 1 + M(h-1) + M(h-2)$ gilt. Können Sie erkennen, warum?

Aus dieser rekursiven Formel können wir eine obere Schranke für $h$ folgendermaßen ableiten:

$$
\begin{align*}
M(h) &= 1 + M(h-1) + M(h-2) \\
M(h-1) &= 1 + M(h-2) + M(h-3) \\
M(h) &= 1 + (1 + M(h-2) + M(h-3)) + M(h-2) & \text{durch Einsetzen}\\
&= 2 + 2M(h-2) + M(h-3) \\
M(h) &> 2M(h-2) \\
& > 4M(h-4) \\
& > 8M(h-6) \\
& > \ldots \\
& > 2^kM(h-2k) \\
& > 2^{\frac{h-1}{2}}M(1) & \text{mit $k = \frac{h-1}{2}$} \\
& > 2^{\frac{h-1}{2}} & \text{da $M(1) = 1$} \\
\log_2 M(h) &> \frac{h-1}{2} \\
h &< 2\log_2 M(h) + 1 \\
h &= O(\log M(h))
\end{align*}
$$

Da $M(h)$ die minimale Anzahl von Knoten in einem AVL-Baum der Höhe $h$ ist, gilt für $N$, die Anzahl der Knoten in *jedem AVL-Baum der Höhe* $h$, dass $N \geq M(h)$, und somit:

$$
h = O(\log N)
$$

Dies ist eine wichtige Schlussfolgerung! Sie bedeutet, dass *alle Operationen*, deren Laufzeit proportional zur Höhe des AVL-Baums ist, eine Laufzeit von $= O(\log n)$ haben.

## 4. Andere optimierte Baumstrukturen

AVL-Bäume sind nicht die einzige Art von optimierten Baum-Datenstrukturen. Weitere Beispiele sind:

- Rot-Schwarz-Bäume  
- Splay-Bäume  
- B-Bäume  

**Rot-Schwarz-Bäume** gelten als balanciert, solange *der längste Pfad von der Wurzel zu einem beliebigen Blatt nicht mehr als doppelt so lang ist wie der kürzeste Pfad*. Dies ist ein schwächeres Kriterium als die AVL-Eigenschaft, aber es ist günstiger zu erhalten und garantiert dennoch eine Höhe von $O(\log N)$ für $N$ Knoten.

**Splay-Bäume** garantieren keine bestimmte Höhe, reorganisieren jedoch den Baum kontinuierlich so, dass die am häufigsten verwendeten Knoten näher an der Wurzel liegen. Dies verbessert die *durchschnittliche* Laufzeitkomplexität der Operationen auf dem Baum.

**B-Bäume** sind für die Festplattenspeicherung optimiert, bei der die Kosten für das Lesen von der Festplatte deutlich höher sind als die Kosten für das Lesen aus dem Speicher. Sie sind so konzipiert, dass die Anzahl der Festplattenzugriffe, die zum Finden eines Knotens im Baum erforderlich sind, minimiert wird. Anstatt eine binäre Baumstruktur zu verwenden, sind B-Bäume *Mehrwegebäume*, bei denen jeder Knoten viele Kinder haben kann.